In [2]:
import pandas as pd
import sys
import os

sibling_dir = os.path.abspath('../01_agentic_rag')

if sibling_dir not in sys.path:
    sys.path.append(sibling_dir)

In [9]:
df_ground_truth = pd.read_csv("data/ground_truth.csv")
df_ground_truth.head(5)

,question,document
0,I just found this course — is it too late to j...,74eb249bbf
1,Can I still take the course if I discovered it...,74eb249bbf
2,Is it okay to join the course after it already...,74eb249bbf
3,"If I join now, can I still get a certificate s...",74eb249bbf
4,What do I need to do to qualify for the certif...,74eb249bbf


In [10]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [6]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [16]:
q=ground_truth[5]
q

{'question': 'I registered for the LLM Zoomcamp, but I still didn’t get any confirmation email — is that normal?',
 'document': '977bf7786c'}

In [17]:
doc_idx[q['document']]

{'id': '977bf7786c',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}

In [18]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [19]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [21]:
answer=assistant.rag(q['question'])

In [22]:
print(answer)

No, that’s normal. You don’t need a confirmation email — you’re already accepted. Registration is only used to gauge interest, and you can start learning and submitting homework while the form is open.


In [23]:
assistant.total_cost()

0.0017565

In [24]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

"You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."

In [26]:
rag_result = {
    "question": q['question'],
    "answer_llm": answer,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'I registered for the LLM Zoomcamp, but I still didn’t get any confirmation email — is that normal?',
 'answer_llm': 'No, that’s normal. You don’t need a confirmation email — you’re already accepted. Registration is only used to gauge interest, and you can start learning and submitting homework while the form is open.',
 'answer_orig': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",
 'document': '977bf7786c'}

In [27]:
# Create a function that processes one ground truth record

def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [ ]:
# running the function on one record
record = generate_rag_answer(q)
record

{'question': 'I registered for the LLM Zoomcamp, but I still didn’t get any confirmation email — is that normal?',
 'answer_llm': 'Yes, that’s normal. You don’t need a confirmation email — you’re accepted. Registration is just to gauge interest, and you can start learning and submitting homework while the form is open.',
 'answer_orig': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",
 'document': '977bf7786c'}

In [ ]:
# reset the usage statistics for the assistant
assistant.reset_usage()

In [31]:
assistant.total_cost()

0.0

In [30]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [32]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer) 

  0%|          | 0/565 [00:00<?, ?it/s]

In [34]:
results[:5]

[{'question': 'I just found this course — is it too late to join now?',
  'answer_llm': 'Yes — you can still join now.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Can I still take the course if I discovered it late?',
  'answer_llm': 'Yes, you can still join the course if you discovered it late. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to join the course after it already started?',
  'answer_llm': 'Yes, you can still join after the course has already started. If you want to receive a certificate,

In [35]:
df_results = pd.DataFrame(results)
df_results.head(5)

,question,answer_llm,answer_orig,document
0,I just found this course — is it too late to j...,Yes — you can still join now.\n\nIf you want a...,"Yes, but if you want to receive a certificate,...",74eb249bbf
1,Can I still take the course if I discovered it...,"Yes, you can still join the course if you disc...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,Is it okay to join the course after it already...,"Yes, you can still join after the course has a...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,"If I join now, can I still get a certificate s...","Yes — if you join now, you can still get a cer...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,What do I need to do to qualify for the certif...,"To qualify for the certificate, you need to fi...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [36]:
df_results.to_csv("data/rag_answers.csv", index=False)

In [37]:
# calculate the total cost of the RAG evaluation

total_cost = assistant.total_cost()
total_cost

0.6211109999999999